# Import Libraries (Initialization)

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, trim


# Read Bronze table.crm_cust_into

In [0]:
df = spark.table('workspace.bronze.crm_cust_info')

# Silver Transformations

## Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

## Normalisation

In [0]:
df = (
    df
    .withColumn(
        "cst_marital_status",
        F.when(F.upper(F.col("cst_marital_status")) == "S", "Single")
        .when(F.upper(F.col("cst_marital_status")) == "M", "Married")
        .otherwise("n/a")
    )
    .withColumn(
        "cst_gndr",
        F.when(F.upper(F.col("cst_gndr")) == "S", "Single")
        .when(F.upper(F.col("cst_gndr")) == "M", "Married")
        .otherwise("n/a")
    )
)

## Remove Records with Missing Customer ID


In [0]:
df = df.filter(col("cst_id").isNotNull())

## Renaming Columns

In [0]:
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_number",
    "cst_firstname": "customer_firstname",
    "cst_lastname": "customer_lastname",
    "cst_marital_status": "marital_status",
    "cst_gndr": "customer_gender",
    "cst_create_date": "created_date"
}

for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

In [0]:
df.display()

# Writing to Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_customers")

# Checking On Silver Table

In [0]:
%sql
SELECT * FROM workspace.silver.crm_customers LIMIT 10